In [30]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.width", 100)
sns.set_theme(style="whitegrid")

FINDINGS = Path("findings.md")
channels = Path("../data/channels.parquet")
clips = Path("../data/clips.parquet")
games = Path("../data/games.parquet")

chdf = pd.read_parquet(channels)
cdf = pd.read_parquet(clips)
gdf = pd.read_parquet(games)

print("channels : ", chdf.shape[0], "rows,", chdf.shape[1], "columns")
print("clips : ", cdf.shape[0], "rows,", cdf.shape[1], "columns")
print("games : ", gdf.shape[0], "rows,", gdf.shape[1], "columns")

channels :  877 rows, 11 columns
clips :  1992 rows, 17 columns
games :  20 rows, 4 columns


### Transforming the data

In [ ]:
# TODO: remove personal data in data acquisition notebook
# remove uncessary columns
# id in chdf matches user_id in cdf, so we keep it.
chdf = chdf.drop(columns=["broadcaster_login", "broadcaster_name"])
cdf = cdf.drop(columns=["creator_id", "broadcaster_name", "creator_name", "thumbnail_url", "url", "embed_url"])
gdf = gdf.drop(columns=["box_art_url"])


# now we transform variables to proper types
chdf['broadcaster_id'] = pd.to_numeric(chdf["broadcaster_id"], errors="coerce").astype("Int64")
chdf['game_id'] = pd.to_numeric(chdf["game_id"], errors="coerce").astype("Int64")
chdf['game_name'] = chdf['game_name'].astype("string")
chdf['broadcaster_language'] = chdf['broadcaster_language'].astype("string")
chdf['title'] = chdf['title'].astype("string")
# Here we parse the tags column, which is a comma-separated string of tags, into a list of strings. We also strip whitespace from each tag.
chdf['tags'] = chdf['tags'].str.split(",").apply(lambda x: [item.strip() for item in x] if isinstance(x, list) else x)
chdf['content_classification_labels'] = chdf['content_classification_labels'].astype("string")


# Clip IDs are strings
cdf["id"] = cdf["id"].astype("string")
cdf["broadcaster_id"] = pd.to_numeric(cdf["broadcaster_id"], errors="coerce").astype("Int64")
cdf["video_id"] = pd.to_numeric(cdf["video_id"], errors="coerce").astype("Int64")
cdf["game_id"] = pd.to_numeric(cdf["game_id"], errors="coerce").astype("Int64")
cdf["language"] = cdf["language"].astype("string")
cdf["title"] = cdf["title"].astype("string")
# Quantitative / Numerical features
cdf["view_count"] = pd.to_numeric(cdf["view_count"], errors="coerce").astype("Int64")
cdf["duration"] = pd.to_numeric(cdf["duration"], errors="coerce").astype("Float64")
# vod_offset is in seconds; Int64 accommodates clips where VOD no longer exists (null)
cdf["vod_offset"] = pd.to_numeric(cdf["vod_offset"], errors="coerce").astype("Int64")
cdf["created_at"] = pd.to_datetime(cdf["created_at"], utc=True)
cdf["is_featured"] = cdf["is_featured"].astype("boolean")

gdf["id"] = pd.to_numeric(gdf["id"], errors="coerce").astype("Int64")
gdf["name"] = gdf["name"].astype("string")
gdf["igdb_id"] = pd.to_numeric(gdf["igdb_id"], errors="coerce").astype("Int64")


print("channels dtypes:")
print(chdf.dtypes.to_string())
print("\nclips dtypes:")
print(cdf.dtypes.to_string())
print("\ngames dtypes:")
print(gdf.dtypes.to_string())

channels dtypes:
broadcaster_id                            Int64
broadcaster_language             string[python]
game_id                                   Int64
game_name                        string[python]
title                            string[python]
delay                                     int64
tags                                    float64
content_classification_labels    string[python]
is_branded_content                         bool

clips dtypes:
id                     string[python]
broadcaster_id                  Int64
video_id                        Int64
game_id                         Int64
language               string[python]
title                  string[python]
view_count                      Int64
created_at        datetime64[ns, UTC]
duration                      Float64
vod_offset                      Int64
is_featured                   boolean

games dtypes:
id                  Int64
name       string[python]
igdb_id             Int64


In [ ]:
# estimators of quantitative vars
print("delay descriptors : ", chdf['delay'].describe().round(2).to_string())
print("view_count descriptors : ", cdf['view_count'].describe().round(2).to_string())


delay descriptors :  count    877.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
view_count descriptors :  count      1992.0
mean      8266.01
std      14121.44
min          80.0
25%        936.75
50%        3672.5
75%       9748.25
max      215980.0


### Identifying collumns with missing values : 

In [33]:
for(df, name) in [(chdf, "channels"), (cdf, "clips"), (gdf, "games")]:
    print(f"\n{name} missing values:")
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(missing.to_string())



channels missing values:
game_id      5
tags       877

clips missing values:
video_id      477
vod_offset    477

games missing values:
igdb_id    6


## Plotting Data

## Save Findings.md
